In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from pathlib import Path
import pandas as pd
import shutil
from sklearn.model_selection import train_test_split
import zipfile

In [ ]:
BASE_DIR = Path("/content/drive/MyDrive/Final")
ZIP_224 = BASE_DIR / "preprocessed-data/images_224.zip"
ZIP_512 = BASE_DIR / "preprocessed-data/images_512.zip"
CLASS_CSV = BASE_DIR / "preprocessed-data/classification_labels_224.csv"
BBOX_CSV = BASE_DIR / "preprocessed-data/resized_bboxes_512.csv"

EXTRACT_DIR_224 = Path("/content/images_224")
EXTRACT_DIR_512 = Path("/content/images_512")

EXTRACT_DIR_224.mkdir(exist_ok=True)
EXTRACT_DIR_512.mkdir(exist_ok=True)

In [ ]:
def extract_zip(zip_path, extract_to):
    print(f"Extracting {zip_path} ...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_to)
    print(f"Done extracting {zip_path} to {extract_to}.")

extract_zip(ZIP_224, EXTRACT_DIR_224)
extract_zip(ZIP_512, EXTRACT_DIR_512)

class_df = pd.read_csv(CLASS_CSV)
bbox_df = pd.read_csv(BBOX_CSV)

all_ids = class_df['patientId'].unique()
train_ids, val_ids = train_test_split(all_ids, test_size=0.2, random_state=42)

print(f"Total IDs: {len(all_ids)}, Train: {len(train_ids)}, Val: {len(val_ids)}")

def copy_images(size, extract_dir, train_ids, val_ids):
    train_dir = BASE_DIR / f"train_val_split/train_{size}"
    val_dir = BASE_DIR / f"train_val_split/val_{size}"
    train_dir.mkdir(exist_ok=True)
    val_dir.mkdir(exist_ok=True)

    train_count, val_count = 0, 0

    for pid in train_ids:
        matches = list(extract_dir.rglob(f"{pid}.png"))
        if matches:
            shutil.copy(matches[0], train_dir / f"{pid}.png")
            train_count += 1

    for pid in val_ids:
        matches = list(extract_dir.rglob(f"{pid}.png"))
        if matches:
            shutil.copy(matches[0], val_dir / f"{pid}.png")
            val_count += 1

    print(f"[{size}] Train images: {train_count}, Val images: {val_count}")

In [ ]:
copy_images("224", EXTRACT_DIR_224, train_ids, val_ids)
copy_images("512", EXTRACT_DIR_512, train_ids, val_ids)

class_df_train = class_df[class_df['patientId'].isin(train_ids)]
class_df_val = class_df[class_df['patientId'].isin(val_ids)]
bbox_df_train = bbox_df[bbox_df['patientId'].isin(train_ids)]
bbox_df_val = bbox_df[bbox_df['patientId'].isin(val_ids)]

class_df_train.to_csv(BASE_DIR / "train_val_split/classification_train.csv", index=False)
class_df_val.to_csv(BASE_DIR / "train_val_split/classification_val.csv", index=False)
bbox_df_train.to_csv(BASE_DIR / "train_val_split/bboxes_train.csv", index=False)
bbox_df_val.to_csv(BASE_DIR / "train_val_split/bboxes_val.csv", index=False)


Extracting /content/drive/MyDrive/Final/preprocessed-data/images_224.zip ...
Done extracting /content/drive/MyDrive/Final/preprocessed-data/images_224.zip to /content/images_224.
Extracting /content/drive/MyDrive/Final/preprocessed-data/images_512.zip ...
Done extracting /content/drive/MyDrive/Final/preprocessed-data/images_512.zip to /content/images_512.
Total IDs: 32021, Train: 25616, Val: 6405


KeyboardInterrupt: 

Export as zips


In [ ]:
BASE_DIR = Path("/content/drive/MyDrive/Final")
ZIP_224 = BASE_DIR / "preprocessed-data/images_224.zip"
ZIP_512 = BASE_DIR / "preprocessed-data/images_512.zip"
CLASS_CSV = BASE_DIR / "preprocessed-data/classification_labels_224.csv"
BBOX_CSV = BASE_DIR / "preprocessed-data/resized_bboxes_512.csv"

EXTRACT_DIR_224 = Path("/content/images_224")
EXTRACT_DIR_512 = Path("/content/images_512")
EXTRACT_DIR_224.mkdir(exist_ok=True)
EXTRACT_DIR_512.mkdir(exist_ok=True)

In [ ]:
def extract_zip(zip_path, extract_to):
    print(f"Extracting {zip_path} ...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_to)
    print(f"Done extracting {zip_path} to {extract_to}.")

class_df = pd.read_csv(CLASS_CSV)
bbox_df = pd.read_csv(BBOX_CSV)

all_ids = class_df['patientId'].unique()
train_ids, val_ids = train_test_split(all_ids, test_size=0.2, random_state=42)
print(f"Total IDs: {len(all_ids)}, Train: {len(train_ids)}, Val: {len(val_ids)}")

def zip_split(extract_dir, ids_list, zip_path):
    total = len(ids_list)
    count = 0
    missing = 0

    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for i, pid in enumerate(ids_list, 1):
            matches = list(extract_dir.rglob(f"{pid}.png"))

            if matches:
                zipf.write(matches[0], arcname=matches[0].name)
                count += 1
            else:
                missing += 1
                print(f"Missing image: {pid}.png")

            # Progress update every 100 items
            if i % 100 == 0 or i == total:
                print(f"  Progress: {i}/{total} ({i/total*100:.1f}%) "
                      f" | Added: {count} | Missing: {missing}")

    print(f"Total IDs: {total}")
    print(f"Images added: {count}")
    print(f"Missing images: {missing}\n")

In [ ]:
OUTPUT_DIR = BASE_DIR / "train_val_split"
OUTPUT_DIR.mkdir(exist_ok=True)

zip_split(EXTRACT_DIR_224, train_ids, OUTPUT_DIR / "train_224.zip")
zip_split(EXTRACT_DIR_224, val_ids, OUTPUT_DIR / "val_224.zip")
zip_split(EXTRACT_DIR_512, train_ids, OUTPUT_DIR / "train_512.zip")
zip_split(EXTRACT_DIR_512, val_ids, OUTPUT_DIR / "val_512.zip")

class_df[class_df['patientId'].isin(train_ids)].to_csv(OUTPUT_DIR / "classification_train.csv", index=False)
class_df[class_df['patientId'].isin(val_ids)].to_csv(OUTPUT_DIR / "classification_val.csv", index=False)
bbox_df[bbox_df['patientId'].isin(train_ids)].to_csv(OUTPUT_DIR / "bboxes_train.csv", index=False)
bbox_df[bbox_df['patientId'].isin(val_ids)].to_csv(OUTPUT_DIR / "bboxes_val.csv", index=False)

Total IDs: 32021, Train: 25616, Val: 6405

🔧 Creating /content/drive/MyDrive/Final/train_val_split/train_224.zip ...
  Progress: 100/25616 (0.4%)  | Added: 100 | Missing: 0
  Progress: 200/25616 (0.8%)  | Added: 200 | Missing: 0
  Progress: 300/25616 (1.2%)  | Added: 300 | Missing: 0
  Progress: 400/25616 (1.6%)  | Added: 400 | Missing: 0
  Progress: 500/25616 (2.0%)  | Added: 500 | Missing: 0
  Progress: 600/25616 (2.3%)  | Added: 600 | Missing: 0
  Progress: 700/25616 (2.7%)  | Added: 700 | Missing: 0
  Progress: 800/25616 (3.1%)  | Added: 800 | Missing: 0
  Progress: 900/25616 (3.5%)  | Added: 900 | Missing: 0
  Progress: 1000/25616 (3.9%)  | Added: 1000 | Missing: 0
  Progress: 1100/25616 (4.3%)  | Added: 1100 | Missing: 0
  Progress: 1200/25616 (4.7%)  | Added: 1200 | Missing: 0
  Progress: 1300/25616 (5.1%)  | Added: 1300 | Missing: 0
  Progress: 1400/25616 (5.5%)  | Added: 1400 | Missing: 0
  Progress: 1500/25616 (5.9%)  | Added: 1500 | Missing: 0
  Progress: 1600/25616 (6.2%)  